# Prospective performance audit, Scope A (Roadmap Work 7)

This notebook asks how DR-TabPFN performs on a prospective target, using only the last three calendar years of inputs (`last_3`). For each reference input year r from 2007 to 2012, the model is trained on the years r-2 to r. Its death predictions are then scored on the patients of year r and of the later input years, so the change from year to year can be read.

- **Outcome:** death in the calendar year after the input year, from the full raw DEATH events. A patient who died in the input year or earlier is removed, so no row is taken after death.
- **Outcome-valid years:** input years 2007 to 2013 have a full next-year outcome. Input year 2014 is kept only as a representation (deaths in 2015 are censored). Input year 2015 is dropped.
- **Runs on:** Fiji, in `~/SAE-TCAV`. The predictor runs on CPU because multi-year contexts exceed the 8 GB GPU's attention memory, the same as the same-year builds. This notebook is executed in place only after the user approves the run. Outputs go to `stats/prospective_target_audit/scope_a/`.

In [1]:
import gc
import json
import subprocess
import time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import prospective_audit as audit

STARTED_UTC = datetime.now(timezone.utc).isoformat()
RUN_START = time.perf_counter()

POPULATION_PATH = Path("stats/temporal_robustness/_population/prepared.pkl")
POPULATION_SHA256 = "c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7"
EVENTS_PATH = Path("tidy_event_data.feather")
OUT_DIR = Path("stats/prospective_target_audit/scope_a")
FIGURE_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_YEARS = (2007, 2008, 2009, 2010, 2011, 2012)
TIMING_REFERENCE_YEAR = 2009
PREDICTOR_DEVICE = "cpu"
PREDICTOR_THREADS = 32  # matches joblib_workers of the same-year builds
torch.set_num_threads(PREDICTOR_THREADS)
SCALINGS = {"consistent": True, "historical": False}
PRIMARY_SCALING = "consistent"
EXPECTED_FEATURES = 500
ROLE_NAMES = ("tabpfn_context", "sae_discovery", "rule_discovery", "cav_fit", "concept_selection", "evaluation",
              "fitting")  # "fitting" is the pilot entrant role (patients first seen after r)
BASELINE_MODELS = ("logistic", "xgboost", "total_count", "last_month", "active_months")
RESIDUAL_MODELS = ("last_month", "active_months")
BATCH_ROWS = 256
GRADIENT_BATCH_ROWS = 512
PARITY_ROWS = 32
FINITE_DIFFERENCE_ROWS = 16
PARITY_TOLERANCE = 1e-4
PROBABILITY_ATOL = 1e-5
PROBABILITY_RTOL = 1e-4
IDENTITY_TOLERANCE = 1e-5
FINITE_DIFFERENCE_LABEL = "sus_last_layer_fd"

checks = []


def record_check(check, measured, rule, passed, reference_year="all", scaling="none"):
    checks.append({"check": check, "reference_year": str(reference_year), "scaling": scaling,
                   "measured": measured, "rule": rule, "passed": bool(passed)})


def input_year_groups(year, reference):
    """Masks over evaluation rows: one group per input year, then all later input years pooled."""
    groups = {str(int(value)): year == value for value in np.unique(year)}
    groups["later_inputs"] = year > reference
    return groups


def stratum_masks(prior_context):
    """All rows, rows whose patient has no earlier context row, rows whose patient has earlier context rows."""
    everyone = np.ones(len(prior_context), dtype=bool)
    return {"all": everyone, "no_prior_context_rows": ~prior_context, "prior_context_rows": prior_context}


def summary_frame(point, draws, **keys):
    """One row per metric: estimate, 95% patient-bootstrap interval, number of finite draws."""
    return audit.summarize_draws(point, draws).reset_index(names="metric").assign(**keys)


print("predictor device:", PREDICTOR_DEVICE, "| threads:", PREDICTOR_THREADS, "| torch:", torch.__version__, "| output:", OUT_DIR)
print("bootstrap draws:", audit.BOOTSTRAP_DRAWS, "| seed:", audit.BOOTSTRAP_SEED)

predictor device: cpu | threads: 32 | torch: 2.1.2+cu121 | output: stats/prospective_target_audit/scope_a
bootstrap draws: 500 | seed: 20261106


## Load the population and build the prospective cohort

Question: which patient-years can be used as prospective rows? The population file is checked against its SHA-256 before it is read. Full-history raw DEATH events give each patient's first death year. A row is kept when the patient is alive at 31 December of the input year, and its outcome is the death in the next calendar year. The cohort checks confirm that no row is taken after death, that input year 2015 is dropped, and that the outcome is defined only for outcome-valid years. Timing features (`last_month`, `active_months`) come from non-death care events and are used only by the CPU baselines and the residual check.

In [2]:
population = audit.load_population(POPULATION_PATH, POPULATION_SHA256)
events = pd.read_feather(EVENTS_PATH)
X_population = population["X"]
first_death_raw = audit.first_death_year_by_row(population, events)
cohort = audit.landmark_cohort(population, first_death_raw)
cohort_checks = audit.cohort_check_rows(cohort)
timing = audit.timing_features(cohort, events).reset_index(drop=True)
bootstrap_universe, bootstrap_counts = audit.patient_draw_counts(cohort["patient_id"])

record_check("population file sha256 equals the approved value", population["fingerprint"]["file_sha256"],
             POPULATION_SHA256, population["fingerprint"]["file_sha256"] == POPULATION_SHA256)
record_check("population feature count", X_population.shape[1], f"= {EXPECTED_FEATURES}",
             X_population.shape[1] == EXPECTED_FEATURES)
for check_row in cohort_checks.to_dict(orient="records"):
    record_check(check_row["check"], check_row["value"], check_row["rule"], check_row["passed"])

cohort_by_year = cohort.groupby("year").agg(rows=("row", "size"), outcome_valid=("outcome_valid", "sum"),
                                            next_year_deaths=("y_next", "sum"))
print("population rows:", population["fingerprint"]["rows"], "| patients:", population["fingerprint"]["patients"])
print(cohort_by_year.to_string())

population rows: 21900 | patients: 5636
      rows  outcome_valid  next_year_deaths
year                                       
2007  1729           1729             127.0
2008  2176           2176             158.0
2009  2227           2227             153.0
2010  2325           2325             188.0
2011  2208           2208             142.0
2012  2250           2250             126.0
2013  2317           2317             155.0
2014  2315              0               0.0


## Quick CPU timing on reference year 2009

Question: how long does one DR-TabPFN fit and capture take on the CPU? The number sets the runtime estimate for the full run. This check fits the consistent-scaling model and captures the evaluation rows only. The loop below rebuilds 2009 in full, so its time is measured again there.

In [3]:
cohort_t, _ = audit.add_roles(cohort, population, TIMING_REFERENCE_YEAR)
context_t = audit.context_mask(cohort_t, TIMING_REFERENCE_YEAR)
evaluation_t = audit.evaluation_mask(cohort_t, TIMING_REFERENCE_YEAR)
row_t = cohort_t["row"].to_numpy()
year_t = cohort_t["year"].to_numpy()

started = time.perf_counter()
fitted_t = audit.fit_dr_tabpfn_context(
    X_population[row_t[context_t]], cohort_t["y_next"].to_numpy(dtype=float)[context_t], year_t[context_t],
    domain_origin=audit.context_first_year(TIMING_REFERENCE_YEAR), device=PREDICTOR_DEVICE, seed=audit.SEED,
    use_context_scaler=True)
captured_t = audit.capture_scored_rows(fitted_t, X_population[row_t[evaluation_t]], year_t[evaluation_t],
                                       batch_size=BATCH_ROWS)
quick_timing = {
    "reference_year": TIMING_REFERENCE_YEAR, "scaling": PRIMARY_SCALING, "predictor_device": PREDICTOR_DEVICE,
    "predictor_threads": PREDICTOR_THREADS,
    "context_rows": int(context_t.sum()), "evaluation_rows": int(evaluation_t.sum()),
    "fit_and_capture_seconds": round(time.perf_counter() - started, 2),
}
print(quick_timing)
del fitted_t, captured_t
gc.collect()

Initialized decoder for standard with (None, 10)  and nout 10


/home/rafaelgomes/SAE-TCAV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'reference_year': 2009, 'scaling': 'consistent', 'predictor_device': 'cpu', 'predictor_threads': 32, 'context_rows': 5023, 'evaluation_rows': 1408, 'fit_and_capture_seconds': 159.66}


6019

## Build each reference year (2007 to 2012)

For each reference year r, the steps are:

1. **Roles:** the frozen patient split for year r (`tabpfn_context`, `sae_discovery`, `rule_discovery`, `cav_fit`, `concept_selection`, `evaluation`). Patients who first appear after r get a fixed role.
2. **Context:** the last_3 window max(2007, r-2) to r. It holds the year-r rows of `tabpfn_context` patients and the earlier rows of every patient in the window.
3. **Fit:** DR-TabPFN is fitted on the context rows. Each context row is labelled with its next-year outcome, so the year-r rows carry labels for year r+1. The label horizon of this model therefore ends on 31 December of r+1. Evaluation patients are held out of the year-r context rows. Their earlier-year rows can still be in the context, which is why results are reported by `prior context rows`. The later input years are the temporal comparison.
4. **Scaling:** consistent context scaling is the primary fit. Historical cached scaling is the sensitivity fit.
5. **Checks:** matched joint parity, capture replay, a finite-difference gradient check, and the identity of actual and held probabilities on reference-year rows. Results go into `checks.csv`.
6. **Scores:** actual probabilities for the evaluation rows of input years r to 2013, each row scored with its own features and its own year input. Held at r is defined per row: the same evaluation row keeps its own features unchanged and is scored with the year input set to r. Every evaluation row has a held probability, including patients who first appear after r. Probabilities on the selection rows set the Death-F1 cut-off.
7. **Last layer:** embeddings and gradients for the evaluation rows (primary fit), used in the geometry step.

In [4]:
builds = {}
build_timing = []
for reference in REFERENCE_YEARS:
    started = time.perf_counter()

    cohort_r, roles = audit.add_roles(cohort, population, reference)
    row = cohort_r["row"].to_numpy()
    year = cohort_r["year"].to_numpy()
    role = cohort_r["role"].to_numpy()
    y_next = cohort_r["y_next"].to_numpy(dtype=float)
    context = audit.context_mask(cohort_r, reference)
    evaluation = audit.evaluation_mask(cohort_r, reference)
    selection = audit.selection_mask(cohort_r, reference)
    sae_rows = (year == reference) & (role == "sae_discovery")
    prior_context = audit.prior_context_counts(cohort_r, context)[evaluation] > 0
    eval_rows, eval_year = row[evaluation], year[evaluation]

    held_year = np.full(len(eval_rows), reference)

    record_check("every row from the reference year onward has a pilot role name (incl. entrant role fitting)",
                 int(np.isin(role[year >= reference], ROLE_NAMES).sum()), f"= {int((year >= reference).sum())}",
                 bool(np.isin(role[year >= reference], ROLE_NAMES).all()), reference)

    context_rows = row[context]
    fit_labels = y_next[context]
    if not np.isfinite(fit_labels).all():
        raise ValueError("context rows must carry a next-year outcome")
    domain_origin = audit.context_first_year(reference)

    per_scaling, primary = {}, {}
    for scaling, use_context_scaler in SCALINGS.items():
        fitted = audit.fit_dr_tabpfn_context(
            X_population[context_rows], fit_labels, year[context], domain_origin=domain_origin, device=PREDICTOR_DEVICE,
            seed=audit.SEED, use_context_scaler=use_context_scaler)
        eval_capture = audit.capture_scored_rows(fitted, X_population[eval_rows], eval_year, batch_size=BATCH_ROWS)
        held_capture = audit.capture_scored_rows(fitted, X_population[eval_rows], held_year, batch_size=BATCH_ROWS)
        selection_probabilities = audit.capture_scored_rows(
            fitted, X_population[row[selection]], year[selection], batch_size=BATCH_ROWS)["probabilities"].astype(float)
        threshold = audit.death_f1_threshold(y_next[selection].astype(np.int64), selection_probabilities)
        same_year = eval_year == reference
        identity_gap = float(np.max(np.abs(eval_capture["probabilities"] - held_capture["probabilities"])[same_year]))

        record_check("context rows used without silent subsampling", fitted["subsample_rows"],
                     f"= {fitted['context_rows']}", fitted["subsample_rows"] == fitted["context_rows"],
                     reference, scaling)
        record_check("held-at-reference identity on reference-year rows (max abs diff)", identity_gap,
                     f"<= {IDENTITY_TOLERANCE:g}", identity_gap <= IDENTITY_TOLERANCE, reference, scaling)

        if scaling == PRIMARY_SCALING:
            parity = audit.parity_check(fitted, X_population[eval_rows[:PARITY_ROWS]], eval_year[:PARITY_ROWS])
            record_check("matched joint execution parity (max abs diff, probability or embedding)",
                         max(parity["proba_max_abs_diff"], parity["emb_max_abs_diff"]),
                         f"<= {PARITY_TOLERANCE:g}", parity["passed"], reference, scaling)
            replay_limit = PROBABILITY_ATOL + PROBABILITY_RTOL * float(np.max(eval_capture["probabilities"]))
            record_check("capture replay error (max abs)", eval_capture["max_replay_error"],
                         f"<= {replay_limit:.2e} (atol 1e-5 + rtol 1e-4 x max probability)",
                         eval_capture["max_replay_error"] <= replay_limit, reference, scaling)
            gradients, fd_report = audit.gradients_with_fd(
                eval_capture, grad_batch_size=GRADIENT_BATCH_ROWS, fd_rows=FINITE_DIFFERENCE_ROWS)
            record_check(FINITE_DIFFERENCE_LABEL + " (max abs error of last-layer gradient)",
                         fd_report["max_absolute_error"], "elementwise rtol 1e-3, atol 1e-5",
                         fd_report["passed"], reference, scaling)
            primary = {
                "embeddings": eval_capture["embeddings"].copy(),
                "gradients": gradients,
                "sae_embeddings": audit.capture_scored_rows(
                    fitted, X_population[row[sae_rows]], year[sae_rows], batch_size=BATCH_ROWS)["embeddings"],
            }

        per_scaling[scaling] = {
            "p_actual": eval_capture["probabilities"].astype(float),
            "p_held": held_capture["probabilities"].astype(float),
            "threshold": float(threshold),
        }
        del fitted, eval_capture
        gc.collect()

    builds[reference] = {
        "reference_year": reference, "cohort": cohort_r, "roles": roles, "context": context,
        "evaluation": evaluation, "selection": selection, "year": eval_year, "label": y_next[evaluation],
        "patient_id": cohort_r["patient_id"].to_numpy()[evaluation], "prior_context": prior_context,
        "base_rate": float(fit_labels.mean()), "per_scaling": per_scaling,
        "counts": {"context_rows": int(context.sum()), "evaluation_rows": int(evaluation.sum()),
                   "evaluation_deaths": int(y_next[evaluation].sum()), "selection_rows": int(selection.sum()),
                   "selection_deaths": int(y_next[selection].sum())},
        **primary,
    }
    build_timing.append({"reference_year": reference, "seconds": round(time.perf_counter() - started, 2)})
    print(reference, builds[reference]["counts"], "| seconds:", build_timing[-1]["seconds"])

{"absolute_error": [4.783272743225098e-06, 2.212822437286377e-06, 1.0870397090911865e-05, 6.332993507385254e-07, 9.164214134216309e-07, 7.018446922302246e-06, 1.5720725059509277e-05, 1.3165175914764404e-05, 1.0602176189422607e-05, 7.599592208862305e-07, 1.5869736671447754e-06, 1.691281795501709e-06, 1.3533979654312134e-05, 4.522502422332764e-06, 2.2351741790771484e-06, 6.459653377532959e-06], "analytical_derivatives": [0.0728246420621872, 0.07623808830976486, 0.07068707793951035, 0.0700874999165535, 0.0686580166220665, 0.06702201068401337, 0.06562687456607819, 0.06910090893507004, 0.07098483294248581, 0.06776677072048187, 0.06338050216436386, 0.07210393995046616, 0.062143679708242416, 0.06456965953111649, 0.0689484179019928, 0.06595303863286972], "atol": 1e-05, "central_derivatives": [0.07282942533493042, 0.07624030113220215, 0.07067620754241943, 0.07008686661720276, 0.06865710020065308, 0.06701499223709106, 0.0656425952911377, 0.06908774375915527, 0.07097423076629639, 0.06776601076126

2007 {'context_rows': 863, 'evaluation_rows': 1894, 'evaluation_deaths': 127, 'selection_rows': 431, 'selection_deaths': 26} | seconds: 102.04


{"absolute_error": [1.0721385478973389e-05, 2.8759241104125977e-06, 6.5267086029052734e-06, 4.127621650695801e-06, 3.986060619354248e-07, 4.16487455368042e-06, 8.959323167800903e-06, 2.2202730178833008e-06, 1.2643635272979736e-05, 3.602355718612671e-06, 4.589557647705078e-06, 2.8312206268310547e-07, 1.3485550880432129e-06, 1.006200909614563e-05, 5.833804607391357e-06, 1.1213123798370361e-06], "analytical_derivatives": [0.06636277586221695, 0.06794567406177521, 0.06289872527122498, 0.06337951123714447, 0.062414657324552536, 0.05837858468294144, 0.0581294484436512, 0.06059779226779938, 0.06176310032606125, 0.05896526202559471, 0.055816650390625, 0.06348221004009247, 0.05485028773546219, 0.05584247037768364, 0.05883263796567917, 0.05548186972737312], "atol": 1e-05, "central_derivatives": [0.06637349724769592, 0.06794854998588562, 0.06290525197982788, 0.06337538361549377, 0.0624142587184906, 0.05838274955749512, 0.0581204891204834, 0.060595571994781494, 0.06177574396133423, 0.0589616596698

2008 {'context_rows': 2815, 'evaluation_rows': 1674, 'evaluation_deaths': 112, 'selection_rows': 546, 'selection_deaths': 47} | seconds: 300.14


{"absolute_error": [2.436339855194092e-06, 7.5660645961761475e-06, 3.7103891372680664e-06, 2.3618340492248535e-06, 1.55717134475708e-06, 4.06801700592041e-06, 2.942979335784912e-07, 2.2687017917633057e-06, 3.606081008911133e-06, 2.1085143089294434e-06, 4.4405460357666016e-06, 1.7881393432617188e-07, 6.332993507385254e-08, 4.719942808151245e-06, 1.4901161193847656e-06, 3.0919909477233887e-06], "analytical_derivatives": [0.05787888914346695, 0.059308972209692, 0.05411358177661896, 0.056733809411525726, 0.05664534121751785, 0.0531599223613739, 0.05274758115410805, 0.05373511090874672, 0.055058553814888, 0.052831195294857025, 0.050140514969825745, 0.05617569386959076, 0.04942261800169945, 0.050477560609579086, 0.05371123552322388, 0.05088288336992264], "atol": 1e-05, "central_derivatives": [0.057881325483322144, 0.059301406145095825, 0.054117292165756226, 0.05673617124557495, 0.056643784046173096, 0.05316399037837982, 0.052747875452041626, 0.053732842206954956, 0.05505494773387909, 0.05282

2009 {'context_rows': 5023, 'evaluation_rows': 1408, 'evaluation_deaths': 99, 'selection_rows': 558, 'selection_deaths': 42} | seconds: 684.97


... (2 outputs left out)

{"absolute_error": [9.499490261077881e-06, 2.518296241760254e-06, 3.6731362342834473e-06, 4.351139068603516e-06, 5.2228569984436035e-06, 6.6086649894714355e-06, 7.450580596923828e-09, 4.511326551437378e-06, 1.4230608940124512e-06, 1.5050172805786133e-06, 5.327165126800537e-06, 3.885477781295776e-06, 3.0249357223510742e-06, 2.332031726837158e-06, 3.3564865589141846e-06, 3.416091203689575e-06], "analytical_derivatives": [0.049164704978466034, 0.04907296597957611, 0.0456080362200737, 0.04801589250564575, 0.04790649563074112, 0.045035310089588165, 0.04510582238435745, 0.046402547508478165, 0.046928904950618744, 0.04507078230381012, 0.04292156547307968, 0.047699522227048874, 0.042405396699905396, 0.04290667921304703, 0.045536454766988754, 0.04407570883631706], "atol": 1e-05, "central_derivatives": [0.049155205488204956, 0.04907548427581787, 0.04561170935630798, 0.04801154136657715, 0.047901272773742676, 0.045028701424598694, 0.045105814933776855, 0.0464070588350296, 0.04692748188972473, 0.0

2011 {'context_rows': 5651, 'evaluation_rows': 863, 'evaluation_deaths': 57, 'selection_rows': 557, 'selection_deaths': 22} | seconds: 890.32


{"absolute_error": [3.1776726245880127e-06, 1.9371509552001953e-07, 2.0265579223632812e-06, 5.502253770828247e-06, 5.662441253662109e-06, 1.6242265701293945e-06, 2.726912498474121e-06, 4.3958425521850586e-07, 4.366040229797363e-06, 1.7881393432617188e-06, 3.0659139156341553e-06, 6.332993507385254e-07, 9.052455425262451e-07, 3.6582350730895996e-06, 2.391636371612549e-06, 4.76837158203125e-06], "analytical_derivatives": [0.04520436003804207, 0.045893147587776184, 0.04092811048030853, 0.04243544116616249, 0.04174381494522095, 0.03882957994937897, 0.038750216364860535, 0.03925754874944687, 0.040013983845710754, 0.0380803644657135, 0.036089036613702774, 0.03965970128774643, 0.035206388682127, 0.035696543753147125, 0.03748296946287155, 0.03626413643360138], "atol": 1e-05, "central_derivatives": [0.04520118236541748, 0.045893341302871704, 0.040930137038230896, 0.04242993891239166, 0.041738152503967285, 0.03882795572280884, 0.03874748945236206, 0.03925710916519165, 0.04000961780548096, 0.03808

2012 {'context_rows': 5650, 'evaluation_rows': 570, 'evaluation_deaths': 37, 'selection_rows': 566, 'selection_deaths': 28} | seconds: 1508.07


## Performance by input year, input-year sensitivity and year-minus-reference

Question: how do the performance measures change from one input year to the next? Each measure is computed on the evaluation rows of each input year, and on all later input years pooled. Death-F1 is the harmonic mean of precision and recall for the death class. Its cut-off is chosen once on the reference-year selection rows and then applied unchanged to every later year. Predictive entropy is the uncertainty of each probability in nats (natural logarithms).

Uncertainty comes from a 500-draw patient bootstrap with seed 20261106. Each draw resamples patients, and every row inherits its patient's draw, so years, models and the actual and held probabilities are compared on the same patients. The sensitivity table is actual minus held at r, on the same evaluation rows. Held at r is each row with its own features and the year input set to r. The two are paired one to one, since every evaluation row has a held probability. The year-minus-reference table compares each input year with the reference year. Results are reported for all rows, for rows without earlier context rows, and for rows with earlier context rows, side by side, without a pass mark.

In [5]:
performance_frames, sensitivity_frames = [], []
for reference, build in builds.items():
    codes = audit.row_patient_codes(build["patient_id"], bootstrap_universe)
    label, year = build["label"], build["year"]
    groups = input_year_groups(year, reference)
    strata = stratum_masks(build["prior_context"])
    for scaling in SCALINGS:
        result = build["per_scaling"][scaling]
        threshold, base_rate = result["threshold"], build["base_rate"]
        actual = {}
        for group, in_group in groups.items():
            for stratum, in_stratum in strata.items():
                keep = in_group & in_stratum
                if not keep.any():
                    continue
                actual[(group, stratum)] = audit.metric_draws(
                    label[keep], result["p_actual"][keep], codes[keep], bootstrap_counts, threshold, base_rate)
                performance_frames.append(summary_frame(
                    *actual[(group, stratum)], reference_year=reference, scaling=scaling,
                    input_year=group, stratum=stratum))
                held = audit.metric_draws(
                    label[keep], result["p_held"][keep], codes[keep], bootstrap_counts, threshold, base_rate)
                sensitivity_frames.append(summary_frame(
                    *audit.difference_draws(*actual[(group, stratum)], *held), reference_year=reference,
                    scaling=scaling, comparison="actual_minus_held_at_reference", input_year=group,
                    stratum=stratum))
        reference_key = str(reference)
        for (group, stratum), values in actual.items():
            if group == reference_key or (reference_key, stratum) not in actual:
                continue
            sensitivity_frames.append(summary_frame(
                *audit.difference_draws(*values, *actual[(reference_key, stratum)]), reference_year=reference,
                scaling=scaling, comparison="input_year_minus_reference_year", input_year=group, stratum=stratum))

longitudinal = pd.concat(performance_frames, ignore_index=True)
sensitivity = pd.concat(sensitivity_frames, ignore_index=True)
longitudinal.to_csv(OUT_DIR / "longitudinal_performance.csv", index=False)
sensitivity.to_csv(OUT_DIR / "year_input_sensitivity.csv", index=False)
print(longitudinal[(longitudinal["metric"] == "auroc") & (longitudinal["stratum"] == "all")
                   & (longitudinal["scaling"] == PRIMARY_SCALING)].to_string(index=False))

metric  estimate   ci_low  ci_high  valid_draws  reference_year    scaling   input_year stratum
 auroc  0.671365 0.568872 0.774722          500            2007 consistent         2007     all
 auroc  0.710717 0.606809 0.807500          500            2007 consistent         2008     all
 auroc  0.734222 0.607564 0.849264          500            2007 consistent         2009     all
 auroc  0.756268 0.651877 0.851252          500            2007 consistent         2010     all
 auroc  0.514080 0.376042 0.637294          500            2007 consistent         2011     all
 auroc  0.589354 0.488993 0.680024          500            2007 consistent         2012     all
 auroc  0.557044 0.420389 0.684135          500            2007 consistent         2013     all
 auroc  0.636955 0.587203 0.682728          500            2007 consistent later_inputs     all
 auroc  0.829659 0.729640 0.925438          500            2008 consistent         2008     all
 auroc  0.730638 0.592641 0.845735      

## CPU baselines and the residual timing check

Question: do simple models, refitted on the same context rows, reach the performance of DR-TabPFN? The baselines are a logistic regression, a boosted-tree classifier (XGBoost), a total-count logistic model, and logistic models on `last_month` and `active_months`. Each baseline gets its own Death-F1 cut-off from its reference-year selection rows.

The residual timing check scores `last_month` and `active_months` alone on the rows that were not used to fit the context, for every input year. If these timing features separate deaths well on their own, the timing of the last care records carries signal about the next year. Its intervals use the same patient bootstrap.

In [6]:
baseline_frames, residual_frames = [], []
for reference, build in builds.items():
    cohort_r = build["cohort"]
    row = cohort_r["row"].to_numpy()
    year = cohort_r["year"].to_numpy()
    outcome = cohort_r["y_next"].to_numpy(dtype=float)
    context, evaluation, selection = build["context"], build["evaluation"], build["selection"]
    X_cohort = X_population[row]
    scores = audit.baseline_scores(X_cohort[context], outcome[context].astype(np.int64), timing[context],
                                   X_cohort, timing)
    base_rate = float(outcome[context].mean())

    codes = audit.row_patient_codes(cohort_r["patient_id"].to_numpy()[evaluation], bootstrap_universe)
    groups = input_year_groups(year[evaluation], reference)
    for name in BASELINE_MODELS:
        threshold = audit.death_f1_threshold(outcome[selection].astype(np.int64), scores[name][selection])
        for group, in_group in groups.items():
            point, draws = audit.metric_draws(
                outcome[evaluation][in_group], scores[name][evaluation][in_group], codes[in_group],
                bootstrap_counts, threshold, base_rate)
            baseline_frames.append(summary_frame(point, draws, reference_year=reference, model=name, input_year=group))

    residual_rows = (~context) & (year >= reference) & ~np.isnan(outcome)
    residual_codes = audit.row_patient_codes(cohort_r["patient_id"].to_numpy()[residual_rows], bootstrap_universe)
    residual_groups = input_year_groups(year[residual_rows], reference)
    for name in RESIDUAL_MODELS:
        threshold = audit.death_f1_threshold(outcome[selection].astype(np.int64), scores[name][selection])
        for group, in_group in residual_groups.items():
            point, draws = audit.metric_draws(
                outcome[residual_rows][in_group], scores[name][residual_rows][in_group],
                residual_codes[in_group], bootstrap_counts, threshold, base_rate)
            residual_frames.append(summary_frame(point, draws, reference_year=reference, model=name,
                                                 input_year=group))

baselines = pd.concat(baseline_frames, ignore_index=True)
residual = pd.concat(residual_frames, ignore_index=True)
baselines.to_csv(OUT_DIR / "baselines.csv", index=False)
residual[residual["metric"].isin(["rows", "deaths", "auroc"])].to_csv(
    OUT_DIR / "residual_leakage_baselines.csv", index=False)
print(residual[(residual["metric"] == "auroc")].to_string(index=False))

metric  estimate   ci_low  ci_high  valid_draws  reference_year         model   input_year
 auroc  0.545505 0.490553 0.593928          500            2007    last_month         2007
 auroc  0.580033 0.550083 0.610971          500            2007    last_month         2008
 auroc  0.565738 0.534758 0.593962          500            2007    last_month         2009
 auroc  0.554547 0.526042 0.583835          500            2007    last_month         2010
 auroc  0.519583 0.482624 0.552114          500            2007    last_month         2011
 auroc  0.527669 0.493703 0.560711          500            2007    last_month         2012
 auroc  0.521030 0.491972 0.552857          500            2007    last_month         2013
 auroc  0.544840 0.532254 0.556307          500            2007    last_month later_inputs
 auroc  0.457942 0.387262 0.517239          500            2007 active_months         2007
 auroc  0.466631 0.421788 0.509974          500            2007 active_months         2008

## Last-layer geometry and the prospective splits file

Question: what does the last layer look like on the evaluation rows (participation ratio, gradient coherence, dimension, and the random-direction TCAV saturation from Work 2)? The standardisation scale comes from the year-r `sae_discovery` embeddings, which are not evaluation rows. The splits file stores the row masks, outcomes, roles, context masks and Death-F1 cut-offs for each reference year, for the later Work 8a step.

In [7]:
geometry_frames = []
for reference, build in builds.items():
    summary = audit.geometry_summary(
        build["embeddings"], build["gradients"], build["per_scaling"][PRIMARY_SCALING]["p_actual"],
        build["sae_embeddings"])
    geometry_frames.append(audit.flatten_geometry(summary).assign(reference_year=reference))
geometry = pd.concat(geometry_frames, ignore_index=True)
geometry.to_csv(OUT_DIR / "geometry_summary.csv", index=False)

audit.export_splits_metadata(
    OUT_DIR / "prospective_splits_and_metadata.npz", population, first_death_raw, cohort,
    {reference: build["roles"] for reference, build in builds.items()},
    {reference: build["context"] for reference, build in builds.items()},
    {reference: build["per_scaling"][PRIMARY_SCALING]["threshold"] for reference, build in builds.items()})
print(geometry.head(12).to_string(index=False))

  section             key          metric       value  reference_year
 spectrum             raw          n_rows 1894.000000            2007
 spectrum             raw    embedding_pr    2.286070            2007
 spectrum             raw  top1_var_share    0.619215            2007
 spectrum             raw  top5_var_share    0.959071            2007
 spectrum             raw top10_var_share    0.983986            2007
 spectrum    standardized          n_rows 1894.000000            2007
 spectrum    standardized    embedding_pr    2.024293            2007
 spectrum    standardized  top1_var_share    0.687331            2007
 spectrum    standardized  top5_var_share    0.945454            2007
 spectrum    standardized top10_var_share    0.978407            2007
coherence raw|probability       rows_used 1894.000000            2007
coherence raw|probability    rows_dropped    0.000000            2007


## Figures, checks table and run record

Figures show AUROC, Brier skill and Death-F1 by input year, with 95% intervals, for each reference year (primary scaling, all rows). A second figure shows the mean predicted probability and the mean predictive entropy in nats. The checks table is written before the notebook decides anything. The run record stores the commit, the predictor device and thread count, the timings and the package versions. If any check failed, the notebook stops with an error after writing both files.

In [8]:
figure_rows = longitudinal[(longitudinal["scaling"] == PRIMARY_SCALING) & (longitudinal["stratum"] == "all")
                           & (longitudinal["input_year"] != "later_inputs")].copy()
figure_rows["input_year"] = figure_rows["input_year"].astype(int)


def plot_metric_panels(metrics, path, title):
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 3.4), constrained_layout=True)
    for axis, metric in zip(axes, metrics):
        for reference in REFERENCE_YEARS:
            part = figure_rows[(figure_rows["reference_year"] == reference) & (figure_rows["metric"] == metric)]
            axis.errorbar(part["input_year"], part["estimate"],
                          yerr=[np.maximum(part["estimate"] - part["ci_low"], 0), np.maximum(part["ci_high"] - part["estimate"], 0)],
                          marker="o", capsize=2, label=str(reference))
        axis.set_title(metric)
        axis.set_xlabel("input year")
    axes[0].legend(title="reference year", fontsize=8)
    fig.suptitle(title)
    fig.savefig(path, dpi=120)
    plt.close(fig)


plot_metric_panels(["auroc", "brier_skill", "death_f1"], FIGURE_DIR / "performance_by_input_year.png",
                   "Prospective performance by input year (consistent scaling, all rows, 95% intervals)")
plot_metric_panels(["mean_probability", "mean_entropy_nats"], FIGURE_DIR / "probability_and_entropy_by_input_year.png",
                   "Mean probability and mean predictive entropy (nats) by input year (consistent scaling)")

checks_table = pd.DataFrame(checks)
checks_table.to_csv(OUT_DIR / "checks.csv", index=False)

def git_text(*args):
    completed = subprocess.run(["git", *args], capture_output=True, text=True, check=True)
    return completed.stdout.strip()


def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


run_record = {
    "work": "prospective-target-audit", "scope": "A", "notebook": "prospective_performance_audit.ipynb",
    "started_utc": STARTED_UTC, "finished_utc": datetime.now(timezone.utc).isoformat(),
    "elapsed_seconds": round(time.perf_counter() - RUN_START, 1),
    "git_commit": git_text("rev-parse", "HEAD"),
    "uncommitted_entries": git_text("status", "--porcelain").splitlines(),
    "population_sha256": population["fingerprint"]["file_sha256"], "predictor_device": PREDICTOR_DEVICE,
    "predictor_threads": PREDICTOR_THREADS,
    "package_versions": {name: package_version(name) for name in
                         ("torch", "numpy", "pandas", "scikit-learn", "tabpfn", "xgboost", "matplotlib")},
    "reference_years": list(REFERENCE_YEARS), "bootstrap": {"draws": audit.BOOTSTRAP_DRAWS,
                                                             "seed": audit.BOOTSTRAP_SEED},
    "quick_timing": quick_timing, "build_timing": build_timing,
    "builds": {str(reference): {"counts": build["counts"],
                                "death_f1_threshold": {s: v["threshold"] for s, v in build["per_scaling"].items()}}
               for reference, build in builds.items()},
    "checks_passed": int(checks_table["passed"].sum()), "checks_total": int(len(checks_table)),
}
run_record["outputs"] = sorted(path.name for path in OUT_DIR.iterdir())
(OUT_DIR / "run_record.json").write_text(json.dumps(run_record, indent=2, sort_keys=True))

print(checks_table.to_string(index=False))
failed = checks_table.loc[~checks_table["passed"].astype(bool), ["check", "reference_year", "scaling"]]
if len(failed):
    raise RuntimeError(f"{len(failed)} check(s) failed, see checks.csv:\n{failed.to_string(index=False)}")
print(f"all {len(checks_table)} checks passed; outputs in {OUT_DIR}")

                                                                                      check reference_year    scaling                                                         measured                                                             rule  passed
                                           population file sha256 equals the approved value            all       none c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7 c8804591c93087241b9533dce77dd54b2f1e77b23a36400527459cfa8f99add7    True
                                                                   population feature count            all       none                                                              500                                                            = 500    True
                                            rows with raw DEATH on or before the input year            all       none                                                                0                                                  